<div dir="rtl" style="text-align:right">
<h1>وزن ثابت، انتخاب متفاوت</h1><p style="text-align:right"><b>پرسش آزمایش:</b> چرا یک Prompt با چند قانون Sampling ادامه‌های متفاوت می‌گیرد؟</p><p style="text-align:right">پیش‌نیاز: <a href="http://127.0.0.1:8000/part-09/chapter-01/54-generate.html"><bdi dir="ltr">54-generate</bdi></a>، <a href="http://127.0.0.1:8000/part-09/chapter-02/55-temperature.html"><bdi dir="ltr">55-temperature</bdi></a>، <a href="http://127.0.0.1:8000/part-09/chapter-02/56-topkp.html"><bdi dir="ltr">56-topkp</bdi></a>، <a href="http://127.0.0.1:8000/part-09/chapter-03/57-prompts.html"><bdi dir="ltr">57-prompts</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، Kernel را Restart و سپس Run All کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">این دفتر مستقل است و مدل کوچک خودش را در ۸۰ گام آموزش می‌دهد؛ به اجرای دفتر آموزش یا Checkpoint آن نیاز ندارد. پس از این مرحله وزن‌ها ثابت می‌مانند. هدف مقایسهٔ قانون انتخاب است، نه رسیدن به متن باکیفیت یا دستیار قابل اعتماد.</p>
</div>

In [ ]:
from torch.utils.data import DataLoader
from mini_gpt.config import ModelConfig
from mini_gpt.data import prepare_corpus
from mini_gpt.dataset import NextTokenDataset
from mini_gpt.model import MiniGPT
from mini_gpt.train import random_batch
from mini_gpt.evaluate import evaluate
train_ids,valid_ids,tokenizer,metadata = prepare_corpus(ROOT/"data"/"sample.txt",train_fraction=0.8)
config = ModelConfig(vocab_size=tokenizer.vocab_size,context_length=16,
                     embedding_dim=32,num_heads=4,num_layers=1,dropout=0.1)
model = MiniGPT(config).cpu()
optimizer = torch.optim.AdamW(model.parameters(),lr=0.003,weight_decay=0.01)
train_data = NextTokenDataset(train_ids,config.context_length)
valid_data = NextTokenDataset(valid_ids,config.context_length)
batch_rng = torch.Generator().manual_seed(18)
prompt_text = "مدل "
prompt = torch.tensor([tokenizer.encode(prompt_text)],dtype=torch.long)
assert 0 not in prompt[0].tolist()
print("Vocabulary:",list(enumerate(tokenizer.id_to_token)))
print("Parameters:",sum(p.numel() for p in model.parameters()))
print("Train/validation windows:",len(train_data),len(valid_data))
print("Validation unknown rate:",metadata["validation_unknown_rate"])


In [ ]:
for step in range(1,81):
    model.train()
    x,y = random_batch(train_data,batch_size=8,generator=batch_rng)
    optimizer.zero_grad(set_to_none=True)
    _,loss = model(x,y)
    if not torch.isfinite(loss):
        raise RuntimeError("Nonfinite Loss")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0,error_if_nonfinite=True)
    optimizer.step()
model.eval()
print("Training finished; last batch Loss:",loss.item())


<div dir="rtl" style="text-align:right">
<h2>گام اول را ثابت نگه داریم</h2><p style="text-align:right">ابتدا یک ورودی و یک مجموعه Logits ثابت را با چند قانون انتخاب می‌سنجیم. پیش‌بینی کنید Greedy، دمای بالاتر، Top-K و Top-P چگونه مجموعهٔ نامزدها و احتمال‌ها را تغییر می‌دهند. Greedy را با temperature=0 پیاده نمی‌کنیم. نمودار فقط هشت Token با بیشترین امتیاز اولیه را نشان می‌دهد؛ مجموع احتمال‌های نمایش‌داده‌شده ممکن است کمتر از یک باشد.</p>
</div>

In [ ]:
from mini_gpt.sampling import sampling_distribution
methods = {
    "greedy": {"greedy":True},
    "temperature=0.7": {"temperature":0.7},
    "temperature=1.3": {"temperature":1.3},
    "top-k=8": {"top_k":8},
    "top-p=0.8": {"top_p":0.8},
}
with torch.no_grad():
    logits = model(prompt[:,-config.context_length:])[0][:,-1,:]
inspect("last-position logits",logits)
shown_ids = logits[0].argsort(descending=True)[:8].tolist()
print("Shown IDs:",shown_ids)
print("Tokens:",[repr(tokenizer.id_to_token[i]) for i in shown_ids])
fig,ax = plt.subplots(figsize=(8,4))
for name,options in methods.items():
    probabilities = sampling_distribution(logits,**options)[0]
    torch.testing.assert_close(probabilities.sum(),torch.tensor(1.))
    print(name,"candidate count:",torch.count_nonzero(probabilities).item())
    ax.plot(range(len(shown_ids)),[probabilities[i].item() for i in shown_ids],"o-",label=name)
ax.set_xticks(range(len(shown_ids)),[str(i) for i in shown_ids])
ax.set(xlabel="Token ID, sorted by raw score",ylabel="Sampling probability")
ax.legend()
plt.show()


<div dir="rtl" style="text-align:right">
<h2>حالا ادامهٔ واقعی تولید کنیم</h2><p style="text-align:right">از generate واقعی استفاده می‌کنیم. دو Seed برای روش‌های تصادفی می‌گذاریم. یکسان‌بودن Seed میان روش‌ها تضمین خروجی یکسان نیست؛ توزیع عوض شده است. بعد از نخستین اختلاف، خود زمینهٔ ادامه هم متفاوت می‌شود.</p>
</div>

In [ ]:
before = {name:value.detach().clone() for name,value in model.state_dict().items()}
for name,options in methods.items():
    for seed in ((23,) if options.get("greedy") else (23,29)):
        torch.manual_seed(seed)
        generated = model.generate(prompt,max_new_tokens=48,**options)
        inspect(name,generated)
        print(name,"seed",seed)
        print(tokenizer.decode(generated[0].tolist()))
assert all(torch.equal(before[name],value) for name,value in model.state_dict().items())
try:
    model.generate(prompt,max_new_tokens=1,temperature=0)
except ValueError as error:
    print("Expected invalid temperature:",error)
else:
    raise AssertionError("Temperature must be positive")


<div dir="rtl" style="text-align:right">
<h2>زمینهٔ بلند کجا بریده می‌شود؟</h2><p style="text-align:right">generate زمینهٔ ورودی Forward را به طول مجاز می‌برد، اما متن برگشتی شامل تمام Prompt و ادامه است. در این پروژه، شمارهٔ موقعیت‌های پنجرهٔ بریده‌شده دوباره از صفر شروع می‌شود؛ جدول موقعیت نیز با همین شماره‌ها خوانده می‌شود.</p>
</div>

In [ ]:
long_prompt = prompt.repeat(1,6)
with torch.no_grad():
    next_logits = model(long_prompt[:,-config.context_length:])[0][:,-1,:]
    expected_id = next_logits.argmax(-1).item()
    generated = model.generate(long_prompt,max_new_tokens=1,greedy=True)
assert generated.shape[1] == long_prompt.shape[1]+1
assert generated[0,-1].item() == expected_id
print("Full prompt length:",long_prompt.shape[1],"model context:",config.context_length)


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین و برداشت:</b> احتمال گام اول، یک ادامهٔ Greedy و دو ادامه برای هر روش تصادفی را مقایسه کنید. کدام تفاوت را می‌توان به Sampling نسبت داد؟ برای داوری دربارهٔ کیفیت، چه نمونه‌ها و معیارهای بیشتری لازم است؟ دما وزن‌ها را آموزش نمی‌دهد و روان‌بودن یا تنوع، درستی پاسخ را تضمین نمی‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2>برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a href="http://127.0.0.1:8000/part-09/chapter-03/57-prompts.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>